# CP1 Week 4 -- Decision Logic & Classification

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Build rule-based classifiers using nested conditionals
2. Use logical operators (`and`, `or`, `not`) for complex decisions
3. Implement `analyze()` that returns labeled outputs
4. Classify data points using multi-condition rules
5. Understand state labeling and categorization

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Rule-Based Classification

A **classifier** takes data and assigns it a label based on rules.
This is one of the most common patterns in engineering:

- Motor state: Normal / Warning / Critical based on temperature and vibration
- Signal quality: Good / Degraded / Lost based on strength and error rate
- Data validity: Valid / Suspect / Invalid based on range and consistency

Think of it like a triage nurse in a hospital: they look at symptoms and
assign a priority level.

### Example 1 -- Simple classifier

In [ ]:
def classify_temperature(temp):
    """Classify a temperature reading into a category."""
    if temp < 0:
        return "freezing"
    elif temp < 15:
        return "cold"
    elif temp < 25:
        return "normal"
    elif temp < 40:
        return "warm"
    else:
        return "hot"

# Test with several values
test_temps = [-10, 5, 22, 35, 50, 0, 14, 25, 39, 40]
print("Temperature Classification:")
for t in test_temps:
    label = classify_temperature(t)
    print(f"  {t:>5} C -> {label}")

**Expected Output:**
```
Temperature Classification:
   -10 C -> freezing
     5 C -> cold
    22 C -> normal
    35 C -> warm
    50 C -> hot
     0 C -> cold
    14 C -> cold
    25 C -> warm
    39 C -> warm
    40 C -> hot
```

### Example 2 -- Multi-sensor classifier

In the real world, decisions depend on MULTIPLE measurements:

In [ ]:
def classify_machine_state(temp, rpm, vibration):
    """Classify machine state from multiple sensor readings.

    Decision logic:
    - CRITICAL: temp > 80 OR vibration > 50 (immediate danger)
    - WARNING:  temp > 60 AND rpm > 3000 (combined stress)
    - CAUTION:  temp > 60 OR rpm > 3000 (one metric elevated)
    - NORMAL:   everything within limits
    """
    if temp > 80 or vibration > 50:
        return "CRITICAL"
    elif temp > 60 and rpm > 3000:
        return "WARNING"
    elif temp > 60 or rpm > 3000:
        return "CAUTION"
    else:
        return "NORMAL"

# Test cases
cases = [
    (25, 1500, 10),    # all normal
    (65, 3500, 20),    # temp AND rpm high
    (65, 1000, 20),    # only temp elevated
    (85, 1000, 10),    # temp critical
    (50, 1000, 60),    # vibration critical
    (50, 3500, 10),    # only rpm elevated
]

print("Machine State Classification:")
print(f"  {'Temp':>5}  {'RPM':>5}  {'Vib':>5}  ->  State")
print(f"  {'----':>5}  {'---':>5}  {'---':>5}  --  -----")
for temp, rpm, vib in cases:
    state = classify_machine_state(temp, rpm, vib)
    print(f"  {temp:>5}  {rpm:>5}  {vib:>5}  ->  {state}")

**Expected Output:**
```
Machine State Classification:
   Temp    RPM    Vib  ->  State
   ----    ---    ---  --  -----
     25   1500     10  ->  NORMAL
     65   3500     20  ->  WARNING
     65   1000     20  ->  CAUTION
     85   1000     10  ->  CRITICAL
     50   1000     60  ->  CRITICAL
     50   3500     10  ->  CAUTION
```

### Why This Matters for Your Pipeline

Your `analyze()` function will classify each data point. For example:
- Robotics track: classify motor state from sensor readings
- Space track: classify star brightness as "transit" or "normal"
- IoT track: classify room conditions as "comfortable" or "alert"


### Try It Yourself #1

In [ ]:
# TODO: Write a function classify_speed(kmh) that returns:
# "stopped" if < 1
# "slow" if 1-30
# "medium" if 31-80
# "fast" if 81-120
# "dangerous" if > 120

def classify_speed(kmh):
    pass  # Replace with your code

# Test it
for speed in [0, 15, 55, 90, 130, 1, 30, 80, 120]:
    label = classify_speed(speed)
    print(f"  {speed:>4} km/h -> {label}")

### Example 3 -- Nested conditionals

Sometimes the first decision splits into sub-decisions:

In [ ]:
def diagnose_motor(temp, rpm, vibration):
    """Two-level diagnosis: first check severity, then cause."""
    # Level 1: Is there a problem?
    if temp > 80 or vibration > 50:
        severity = "CRITICAL"
        # Level 2: What is the cause?
        if temp > 80 and vibration > 50:
            cause = "Both temperature and vibration extreme"
        elif temp > 80:
            cause = "Temperature too high"
        else:
            cause = "Vibration too high"
    elif temp > 60 or rpm > 3000:
        severity = "WARNING"
        if temp > 60:
            cause = "Temperature elevated"
        else:
            cause = "RPM elevated"
    else:
        severity = "NORMAL"
        cause = "All readings within limits"

    return severity, cause

# Test
test_cases = [
    (25, 1500, 10),
    (85, 1000, 10),
    (50, 1000, 60),
    (90, 4000, 55),
    (65, 1500, 20),
    (50, 3500, 10),
]

print("Motor Diagnosis:")
for temp, rpm, vib in test_cases:
    severity, cause = diagnose_motor(temp, rpm, vib)
    print(f"  T={temp:>3}, RPM={rpm:>4}, V={vib:>3} -> [{severity}] {cause}")

**Expected Output:**
```
Motor Diagnosis:
  T= 25, RPM=1500, V= 10 -> [NORMAL] All readings within limits
  T= 85, RPM=1000, V= 10 -> [CRITICAL] Temperature too high
  T= 50, RPM=1000, V= 60 -> [CRITICAL] Vibration too high
  T= 90, RPM=4000, V= 55 -> [CRITICAL] Both temperature and vibration extreme
  T= 65, RPM=1500, V= 20 -> [WARNING] Temperature elevated
  T= 50, RPM=3500, V= 10 -> [WARNING] RPM elevated
```

### Example 4 -- Building a label summary

In [ ]:
def summarize_labels(labels):
    """Count occurrences of each label and show percentages."""
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    total = len(labels)
    print("Label Summary:")
    for label, count in sorted(counts.items()):
        pct = count / total * 100
        bar = "#" * int(pct / 2)
        print(f"  {label:>10}: {count:3d} ({pct:5.1f}%) {bar}")
    return counts

# Test
import random
random.seed(42)
test_labels = []
for _ in range(50):
    r = random.random()
    if r < 0.6:
        test_labels.append("normal")
    elif r < 0.85:
        test_labels.append("warning")
    else:
        test_labels.append("critical")

summarize_labels(test_labels)

The bar chart gives a quick visual sense of the distribution.
This pattern is useful in your `analyze()` function's output.

---
## Part 2: Building `analyze()` with Statistics and Labels

The `analyze()` function is the brain of your pipeline. It takes clean data
and produces:
1. **Summary statistics** (mean, median, std, min, max)
2. **Labels** for each data point (normal, high, low)
3. **Counts** of each category

In [ ]:
def analyze(clean_data, config):
    """Analyze clean data: compute stats and assign labels.

    Args:
        clean_data: list of dicts, each with a 'value' key
        config: dict with optional 'threshold' key

    Returns:
        dict with 'analysis_summary' and 'labels'
    """
    if not clean_data:
        return {"analysis_summary": {"mean": 0, "median": 0, "std": 0}, "labels": []}

    # Extract numeric values
    values = [row["value"] for row in clean_data]
    n = len(values)

    # Compute basic stats
    mean_val = sum(values) / n
    sorted_vals = sorted(values)
    if n % 2 == 0:
        median_val = (sorted_vals[n // 2 - 1] + sorted_vals[n // 2]) / 2
    else:
        median_val = sorted_vals[n // 2]
    variance = sum((x - mean_val) ** 2 for x in values) / n
    std_val = variance ** 0.5

    # Classify each value
    threshold = config.get("threshold", mean_val + 2 * std_val)
    labels = []
    for v in values:
        if v > threshold:
            labels.append("high")
        elif v < mean_val - 2 * std_val:
            labels.append("low")
        else:
            labels.append("normal")

    results = {
        "analysis_summary": {
            "count": n,
            "mean": round(mean_val, 2),
            "median": round(median_val, 2),
            "std": round(std_val, 2),
            "min": min(values),
            "max": max(values),
            "n_high": labels.count("high"),
            "n_low": labels.count("low"),
            "n_normal": labels.count("normal"),
        },
        "labels": labels,
    }

    print(f"Analysis: {n} values, mean={mean_val:.2f}, std={std_val:.2f}")
    print(f"Labels: {labels.count('normal')} normal, "
          + f"{labels.count('high')} high, {labels.count('low')} low")
    return results

# Test
test_data = [
    {"id": i, "value": v}
    for i, v in enumerate([22, 23, 21, 50, 19, 23, 21, 22, 20, 45])
]
config = {"threshold": 40}
results = analyze(test_data, config)
print()
print("Summary:", results["analysis_summary"])

**Expected Output:**
```
Analysis: 10 values, mean=26.60, std=10.30
Labels: 8 normal, 2 high, 0 low

Summary: {'count': 10, 'mean': 26.6, 'median': 22.0, 'std': 10.3, 'min': 19, 'max': 50, 'n_high': 2, 'n_low': 0, 'n_normal': 8}
```

### Common Mistakes with Classification

| Mistake | What happens | Fix |
|---------|-------------|-----|
| Overlapping ranges | Multiple labels match | Use `elif`, not multiple `if` |
| Wrong operator order | `and`/`or` confusion | Use parentheses: `(a > 5) and (b < 10)` |
| Not handling edge values | Boundary values misclassified | Test with exact boundary values |
| No default case | Some inputs get no label | Always have an `else` |

### Try It Yourself #2

In [ ]:
# TODO: Create a data quality classifier.
# Given a row with value, label it:
# - "missing" if value is None or ""
# - "invalid" if value cannot be converted to float
# - "outlier" if value < -100 or > 1000
# - "suspect" if value < 0
# - "good" otherwise

def classify_quality(value):
    pass  # your code here

test_values = [25.0, None, "", "abc", -5, -150, 1500, 0, 99.9]
for v in test_values:
    label = classify_quality(v)
    print(f"  {str(v):>8} -> {label}")

### Why This Matters for Your Pipeline

Classification is the core of `analyze()`. Your pipeline does not just compute
numbers -- it assigns **meaning** to those numbers. Is this reading normal or
abnormal? Is this trend rising or falling? Is this data point an outlier?

These labels become:
- Rows in your report ("5 CRITICAL events detected")
- Colors on your plot (red = critical, green = normal)
- Decisions in your pipeline (skip outliers, flag warnings)

---
## Key Takeaways -- Week 4

1. **Classifiers** assign labels to data based on rules
2. **Multi-condition** decisions use `and`, `or`, `not`
3. **`analyze()`** computes statistics AND classifies each data point
4. **Order matters** in elif chains -- first True condition wins
5. Labels let you count and filter categories downstream
6. **Boundary testing** ensures edge values are handled correctly

---
## Part 3: Practical Classification Patterns

### Pattern 1: Multi-level classification with scores

In [ ]:
def calculate_risk_score(temp, rpm, vibration):
    """Calculate a numeric risk score from 0-100, then classify."""
    score = 0

    # Temperature contribution (0-40 points)
    if temp > 80:
        score += 40
    elif temp > 60:
        score += 25
    elif temp > 40:
        score += 10

    # RPM contribution (0-30 points)
    if rpm > 3000:
        score += 30
    elif rpm > 2000:
        score += 15

    # Vibration contribution (0-30 points)
    if vibration > 50:
        score += 30
    elif vibration > 30:
        score += 15

    # Classify based on total score
    if score >= 70:
        level = "CRITICAL"
    elif score >= 40:
        level = "WARNING"
    elif score >= 15:
        level = "CAUTION"
    else:
        level = "NORMAL"

    return score, level

# Test
cases = [(25, 1500, 10), (65, 2500, 35), (85, 3500, 55), (45, 1800, 20)]
print("Risk Assessment:")
print(f"  {'Temp':>5} {'RPM':>5} {'Vib':>5}  Score  Level")
print(f"  {'----':>5} {'---':>5} {'---':>5}  -----  -----")
for t, r, v in cases:
    score, level = calculate_risk_score(t, r, v)
    print(f"  {t:>5} {r:>5} {v:>5}  {score:>5}  {level}")

**Expected Output:**
```
Risk Assessment:
   Temp   RPM   Vib  Score  Level
   ----   ---   ---  -----  -----
     25  1500    10      0  NORMAL
     65  2500    35     40  WARNING
     85  3500    55    100  CRITICAL
     45  1800    20     10  NORMAL
```

### Pattern 2: Trend detection

In [ ]:
def detect_trend(values):
    """Detect if the last 3+ values show a trend."""
    if len(values) < 3:
        return "insufficient data"

    last_3 = values[-3:]

    if last_3[0] < last_3[1] < last_3[2]:
        return "rising"
    elif last_3[0] > last_3[1] > last_3[2]:
        return "falling"
    else:
        # Check for stability: all within 10% of mean
        avg = sum(last_3) / len(last_3)
        if avg == 0:
            return "stable"
        max_deviation = max(abs(v - avg) / abs(avg) for v in last_3)
        if max_deviation < 0.1:
            return "stable"
        return "volatile"

# Test
test_series = [
    [10, 20, 30],
    [30, 20, 10],
    [25, 25, 26],
    [10, 50, 20],
    [5],
]

for series in test_series:
    trend = detect_trend(series)
    print(f"  {str(series):>20} -> {trend}")

**Expected Output:**
```
          [10, 20, 30] -> rising
          [30, 20, 10] -> falling
          [25, 25, 26] -> stable
          [10, 50, 20] -> volatile
                   [5] -> insufficient data
```

### Pattern 3: Putting labels back into data

In [ ]:
def label_data(data, config):
    """Add a 'label' field to each row based on its value."""
    threshold = config.get("threshold", 50)
    labeled = []

    for row in data:
        value = row.get("value", 0)
        if value > threshold * 1.5:
            label = "critical"
        elif value > threshold:
            label = "high"
        elif value > threshold * 0.5:
            label = "medium"
        else:
            label = "low"

        labeled.append({**row, "label": label})

    return labeled

# Test
test_data = [
    {"id": 1, "value": 10},
    {"id": 2, "value": 30},
    {"id": 3, "value": 55},
    {"id": 4, "value": 80},
]

labeled = label_data(test_data, {"threshold": 50})
for row in labeled:
    print(f"  id={row['id']}, value={row['value']:>3}, label={row['label']}")

**Expected Output:**
```
  id=1, value= 10, label=low
  id=2, value= 30, label=medium
  id=3, value= 55, label=high
  id=4, value= 80, label=critical
```

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What label does classify_temperature(25) return?
# Answer:

# R2: What is the difference between "and" and "or"?
# Answer:

# R3: If mean=20 and std=5, what is mean + 2*std?
# Answer:

# R4: In an elif chain, how many blocks can execute?
# Answer:

### Practice (P1-P5)

In [ ]:
# P1: Write classify_speed(kmh):
# "stopped" (<1), "slow" (1-30), "medium" (31-80), "fast" (81-120), "dangerous" (>120)
# Test with 10 different values.


In [ ]:
# P2: Classify these 10 readings using classify_temperature():
readings = [5, 18, -3, 30, 42, 0, 15, 25, 38, 22]


In [ ]:
# P3: Write count_by_label(values, labels) that returns a dict
# counting items in each category. Example: {"normal": 5, "high": 3, "low": 2}


In [ ]:
# P4: Write analyze() for your track. Compute at least 5 metrics
# and classify each data point.


In [ ]:
# P5: Create a "status dashboard" that takes a list of readings
# and prints a formatted summary with label counts and percentages.


### Challenge (C1-C3)

In [ ]:
# C1: Implement a "traffic light" classifier:
# Given (temp, pressure, humidity), return "green", "yellow", or "red"
# Document your threshold choices.


In [ ]:
# C2: Write a trend detector: given 5+ values, classify as
# "rising", "falling", "stable", or "volatile"


In [ ]:
# C3: Create a multi-level classifier that uses 4+ input variables
# and produces 5+ output categories. Draw the decision tree as comments.


### Mini-Project

In [ ]:
# M1: Sensor Alert System
# Given 20 sensor readings with temp, rpm, and vibration,
# classify each as NORMAL/CAUTION/WARNING/CRITICAL.
# Print a formatted alert report showing:
# - Each reading with its classification
# - Summary counts
# - Percentage in each category
# - List of all CRITICAL readings


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)